## Loading the Dataset

In [1]:
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD


In [2]:

recipe2M = pd.read_csv('/kaggle/input/recipes/recipes_data.csv')

In [3]:
recipe2M.head()

,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


## Removing unnecessary columns & nulls

In [4]:
recipe2M_cleaned=recipe2M.drop(columns=['link', 'source', 'site'], inplace=False)
recipe2M_cleaned.dropna()

,title,ingredients,directions,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p..."
...,...,...,...,...
2231137,Sunny's Fake Crepes,"[""1/2 cup chocolate hazelnut spread (recommend...","[""Spread hazelnut spread on 1 side of each tor...","[""chocolate hazelnut spread"", ""marshmallows"", ..."
2231138,Devil Eggs,"[""1 dozen eggs"", ""1 paprika"", ""1 salt and pepp...","[""Boil eggs on medium for 30mins."", ""Then cool...","[""choice"", ""miracle whip"", ""eggs"", ""relish"", ""..."
2231139,Extremely Easy and Quick - Namul Daikon Salad,"[""150 grams Daikon radish"", ""1 tbsp Sesame oil...","[""Julienne the daikon and squeeze out the exce...","[""soy sauce"", ""radish"", ""white sesame seeds"", ..."
2231140,Pan-Roasted Pork Chops With Apple Fritters,"[""1 cup apple cider"", ""6 tablespoons sugar"", ""...","[""In a large bowl, mix the apple cider with 4 ...","[""apple cider"", ""egg"", ""sugar"", ""freshly groun..."


## Removing recipes with directions contatining the word "step"

In [5]:

recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned['directions'].str.contains('step', case=False, na=False)]
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with at most 1 ingredient

In [6]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['ingredients'].apply(lambda x: len([i for i in x if i.strip()]) <= 1)].index, inplace=True)
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with instructions less than 10 characters

In [7]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['directions'].apply(lambda x: not all(len(i.strip()) < 10 for i in x if i.strip()))].index, inplace=True)
recipe2M_cleaned['title'].count()

2206617

## Removing recipes with title less than 4 characters 

In [8]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['title'].apply(lambda x: len(str(x)) < 4 if pd.notnull(x) else False)].index, inplace=True)
recipe2M_cleaned['title'].count()

2206372

## Extract raw ingredients

In [9]:
recipes = recipe2M_cleaned

In [10]:
# Tokenize a string into words
recipes['tokens'] = recipes['NER'].apply(word_tokenize)


In [11]:
#adding customized stop words
irrelevant_words = {
    'fresh', 'frozen', 'thawed', 'raw', 'grated', 'diced', 'chopped', 'minced',
    'powdered', 'sliced', 'ground', 'cooked', 'boiled', 'roasted', 'steamed',
    'baked', 'fried', 'toasted', 'crushed', 'peeled', 'skinned', 'shredded',
    'melted', 'whipped', 'pinch', 'dash', 'handful', 'cup', 'tablespoon',
    'teaspoon', 'liter', 'ml', 'oz', 'lb', 'gram', 'kg', 'quart', 'optional',
    'to taste', 'as needed', 'prepared', 'ready-made', 'store-bought', 'homemade',
    'pre-cooked', 'large', 'small', 'medium', 'whole', 'half', 'quartered',
    'extra', 'light', 'dark', 'white', 'black', 'red', 'green', 'yellow',
    'brown', 'golden', 'sweet', 'bitter', 'spicy', 'mild', 'hot', 'cold',
    'water', 'broth', 'stock', 'sauce', 'seasoning', 'marinade','bite','size'
}
stop_words = set(stopwords.words('english'))
stop_words.update(irrelevant_words)

In [12]:
lemmatizer = WordNetLemmatizer()

# Function to lemmatize nouns
def lemmatize(word, pos):
    if pos.startswith('NN'):  
        return lemmatizer.lemmatize(word, pos='n')
    else:
        return word  


In [14]:
import nltk
nltk.download('wordnet', download_dir='/kaggle/working/nltk_data')
nltk.download('omw-1.4', download_dir='/kaggle/working/nltk_data')  # Optional WordNet data
nltk.download('averaged_perceptron_tagger', download_dir='/kaggle/working/nltk_data')
nltk.download('stopwords', download_dir='/kaggle/working/nltk_data')


[nltk_data] Downloading package wordnet to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [15]:
import nltk
import os
import zipfile

# Ensure the custom NLTK path is set
nltk.data.path.append('/kaggle/working/nltk_data')

# Path to the WordNet zip file and the extract path
zip_path = '/kaggle/working/nltk_data/corpora/wordnet.zip'
extract_path = '/kaggle/working/nltk_data/corpora/'

# Extract WordNet data if it exists
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("WordNet data extracted successfully.")
else:
    print("WordNet zip file not found.")

# Verify WordNet path
wordnet_path = '/kaggle/working/nltk_data/corpora/wordnet'
print("WordNet path exists:", os.path.exists(wordnet_path))

# Load WordNet and Stopwords
nltk.download('wordnet', download_dir='/kaggle/working/nltk_data')  # Ensure WordNet is present
nltk.download('averaged_perceptron_tagger', download_dir='/kaggle/working/nltk_data')  # For POS tagging
nltk.download('stopwords', download_dir='/kaggle/working/nltk_data')  # Stopwords

# Apply stopwords and lemmatization
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

# Ensure WordNet is loaded
wordnet.ensure_loaded()

# Initialize stop words and lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Function to convert POS tags to WordNet POS tags
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

WordNet data extracted successfully.
WordNet path exists: True
[nltk_data] Downloading package wordnet to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /kaggle/working/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
# Check if WordNet data exists in the extracted location
print("WordNet path exists:", os.path.exists('/kaggle/working/nltk_data/corpora/wordnet'))


WordNet path exists: True


In [18]:
#apply stop words removal and lemmatization
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: [lemmatize(word.lower(), tag) for word, tag in nltk.pos_tag(x) if word.isalnum() and word.lower() not in stop_words]
)

In [19]:
#filter uninque ingredients
recipes['tokens'] = recipes['tokens'].apply(
    lambda x: list(set(x))
)

In [20]:
print(recipes['tokens'])

0          [size, nut, vanilla, biscuit, brown, butter, m...
1          [soup, mushroom, cream, sour, breast, beef, ch...
2          [cheese, cream, butter, powder, garlic, salt, ...
3          [soup, gravy, mushroom, cheese, cream, chicken...
4          [crumb, chocolate, graham, butter, peanut, chi...
                                 ...                        
2231136    [sweet, potato, coconut, carrot, butter, powde...
2231137    [chocolate, marshmallow, butter, hazelnut, spr...
2231139    [white, seed, oil, soy, radish, salt, sesame, ...
2231140    [chop, pork, thyme, shallot, horseradish, bran...
2231141    [romano, white, veal, garlic, sausage, red, sa...
Name: tokens, Length: 2206373, dtype: object


In [21]:
#adding ids to recipes
recipes['recipe_id'] = recipes.index + 1

In [22]:
#reorder the columns
columns = ['recipe_id'] + [col for col in recipes.columns if col != 'recipe_id']
recipes = recipes[columns]

In [23]:
recipes.rename(columns={'tokens': 'raw_ingredients'}, inplace=True)

In [24]:
recipes.head()

,recipe_id,title,ingredients,directions,NER,raw_ingredients
0,1,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""...","[size, nut, vanilla, biscuit, brown, butter, m..."
1,2,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream...","[soup, mushroom, cream, sour, breast, beef, ch..."
2,3,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[cheese, cream, butter, powder, garlic, salt, ..."
3,4,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c...","[soup, gravy, mushroom, cheese, cream, chicken..."
4,5,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p...","[crumb, chocolate, graham, butter, peanut, chi..."


# Extract Cooking Methods

In [25]:
recipes.head()

,recipe_id,title,ingredients,directions,NER,raw_ingredients
0,1,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""...","[size, nut, vanilla, biscuit, brown, butter, m..."
1,2,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream...","[soup, mushroom, cream, sour, breast, beef, ch..."
2,3,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[cheese, cream, butter, powder, garlic, salt, ..."
3,4,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c...","[soup, gravy, mushroom, cheese, cream, chicken..."
4,5,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p...","[crumb, chocolate, graham, butter, peanut, chi..."


In [26]:
recipes['directions']

0          ["In a heavy 2-quart saucepan, mix brown sugar...
1          ["Place chipped beef on bottom of baking dish....
2          ["In a slow cooker, combine all ingredients. C...
3          ["Boil and debone chicken.", "Put bite size pi...
4          ["Combine first four ingredients and press in ...
                                 ...                        
2231136    ["Cook the onion in butter in a medium saucepa...
2231137    ["Spread hazelnut spread on 1 side of each tor...
2231139    ["Julienne the daikon and squeeze out the exce...
2231140    ["In a large bowl, mix the apple cider with 4 ...
2231141    ["Preheat the oven to 350.", "In a bowl, mix t...
Name: directions, Length: 2206373, dtype: object

In [27]:
cooking_methods_glossary = [
    "bake", "steam", "fry", "grill", "roast", "boil", "sauté", "poach", "broil", "braise",
    "stew", "smoke", "microwave", "blanch", "deep-fry", "barbecue", "sear", "pressure-cook",
    "simmer", "stir-fry","baste","batter","beat","blend","carmelize","chop","cream","cube",
    "cure","dice","dissolve","drain","fold","granish","grate","grease","julienne","knead",
    "marinate","mash","mince","parboil","pare","peel","pinch","pit","plump","preheat","puree",
    "reduce","saute","scald","sear","shred","sift","skim","slice","thaw","toss","whip"
]

In [28]:
stop_words = set(stopwords.words('english'))

In [29]:
cooking_methods = []

for directions in recipes['directions']:
    if pd.isna(directions):
        cooking_methods.append(None)
    else:
        # Tokenize words
        words = word_tokenize(directions.lower())
        # Remove stopwords and non-alphabetic tokens
        filtered_words = [word for word in words if word not in stop_words and word.isalpha()]

        methods = set(filtered_words).intersection(cooking_methods_glossary)

        cooking_methods.append(list(methods))

recipes['cooking_methods'] = cooking_methods

In [30]:
recipes['cooking_methods'].head(20)

0                                      [boil]
1                               [bake, cream]
2                                          []
3                         [bake, cream, boil]
4                                          []
5     [boil, cream, drain, grease, microwave]
6                               [cream, beat]
7                                      [bake]
8                                    [simmer]
9                         [whip, chop, drain]
10                    [drain, fold, dissolve]
11                                         []
12                 [barbecue, microwave, fry]
13                             [cream, drain]
14                                         []
15                              [boil, cream]
16                                     [bake]
17                                     [toss]
18                                    [cream]
19                               [sift, bake]
Name: cooking_methods, dtype: object

## Clean rating dataset

In [31]:
train_rating= pd.read_csv('/kaggle/input/train-rating/core-data-train_rating.csv')
test_rating = pd.read_csv('/kaggle/input/test-rating/core-data-test_rating.csv')

In [32]:
len(train_rating['recipe_id'].unique())

29093

In [33]:
#mapping train data with correct recipe id
unique_old_ids = sorted(train_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist() 

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
train_rating['recipe_id'] = train_rating['recipe_id'].map(mapping)

In [34]:
#mapping test data with correct recipe id
unique_old_ids = sorted(test_rating['recipe_id'].unique())
new_ids = recipes['recipe_id'].tolist()  

mapping = {old: new for old, new in zip(unique_old_ids, new_ids)}
test_rating['recipe_id'] = test_rating['recipe_id'].map(mapping)

In [35]:
train_rating.drop(columns=['dateLastModified'], inplace=True)
test_rating.drop(columns=['dateLastModified'], inplace=True)

In [36]:
train_rating.head()

,user_id,recipe_id,rating
0,5215572,9088,5
1,5215572,25264,4
2,5215572,9138,5
3,3622615,17758,4
4,1313770,16467,5


In [37]:
recipes.to_csv('cleanedrecipes.csv', index=False)

In [38]:
train_rating.to_csv('cleanedTrainRating.csv',index=False)
test_rating.to_csv('cleanedTestRating.csv',index=False)

In [2]:
cleaned_recipes = pd.read_csv('/kaggle/input/cleanedr/cleanedrecipes.csv')
cleaned_Trainrating = pd.read_csv('/kaggle/input/ratingss/cleanedTrainRating.csv')
cleaned_Testrating = pd.read_csv('/kaggle/input/ratingss/cleanedTestRating.csv')

In [3]:
len(cleaned_Testrating['recipe_id'].unique())

37342

In [4]:
cleaned_Testrating

,user_id,recipe_id,rating
0,5215572,13179,5
1,5215572,11034,4
2,5215572,5097,5
3,3622615,11034,5
4,1313770,13227,5
...,...,...,...
283435,3023108,114,3
283436,3023108,7866,5
283437,3023108,3739,4
283438,3023108,33228,4


# Models

## Content Based

In [6]:
import ast  
from sklearn.feature_extraction.text import TfidfVectorizer

def safe_eval(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except:
        return [] 

cleaned_recipes['raw_ingredients'] = cleaned_recipes['raw_ingredients'].apply(safe_eval)
cleaned_recipes['cooking_methods'] = cleaned_recipes['cooking_methods'].apply(safe_eval)

cleaned_recipes['profile'] = cleaned_recipes['raw_ingredients'].apply(lambda x: ' '.join(x)) + ' ' + \
                             cleaned_recipes['cooking_methods'].apply(lambda x: ' '.join(x) if x else '')

# Remove any leading/trailing spaces
cleaned_recipes['profile'] = cleaned_recipes['profile'].str.strip()

print(cleaned_recipes[['profile']].head())

vectorizer = TfidfVectorizer()
recipe_profiles = vectorizer.fit_transform(cleaned_recipes['profile'])

print("Unique words count:", len(vectorizer.get_feature_names_out()))


                                             profile
0  size nut vanilla biscuit brown butter milk sug...
1  soup mushroom cream sour breast beef chicken b...
2  cheese cream butter powder garlic salt frozen ...
3  soup gravy mushroom cheese cream chicken shred...
4  crumb chocolate graham butter peanut chip powd...
Unique words count: 26749


In [14]:
import ast
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

# ---------- Step 1: Ensure the 'profile' column is valid ----------
def safe_eval(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except Exception:
        return []

cleaned_recipes['raw_ingredients'] = cleaned_recipes['raw_ingredients'].apply(
    lambda x: safe_eval(x) if not isinstance(x, list) else x)
cleaned_recipes['cooking_methods'] = cleaned_recipes['cooking_methods'].apply(
    lambda x: safe_eval(x) if not isinstance(x, list) else x)

cleaned_recipes['profile'] = (
    cleaned_recipes['raw_ingredients'].apply(lambda x: ' '.join(x)) + ' ' +
    cleaned_recipes['cooking_methods'].apply(lambda x: ' '.join(x) if x else '')
).str.strip()


# ---------- Step 2: Prepare Training Data ----------
# Use only recipes that appear in cleaned_Trainrating
train_recipes = cleaned_recipes[cleaned_recipes['recipe_id'].isin(cleaned_Trainrating['recipe_id'])]

# ---------- Step 3: Create TF-IDF Vectors for Training Recipes ----------
vectorizer = TfidfVectorizer(min_df=1, max_df=1.0, stop_words=None)
recipe_profiles = vectorizer.fit_transform(train_recipes['profile'])

# ---------- Step 4: Create User Profiles Based on Training Data ----------
# The user profile is a weighted average of the TF-IDF vectors for recipes the user rated.
user_profiles = {}
for user_id in cleaned_Trainrating['user_id'].unique():
    user_ratings = cleaned_Trainrating[cleaned_Trainrating['user_id'] == user_id]
    rated_recipe_ids = user_ratings['recipe_id']
    rated_recipes = train_recipes[train_recipes['recipe_id'].isin(rated_recipe_ids)]
    rated_profiles = recipe_profiles[train_recipes['recipe_id'].isin(rated_recipe_ids)]
    user_ratings_values = user_ratings['rating'].values
    if user_ratings_values.sum() == 0:
        continue 
    user_profile = rated_profiles.T.dot(user_ratings_values) / user_ratings_values.sum()
    user_profiles[user_id] = np.asarray(user_profile).flatten()

# ---------- Step 5: Prepare Testing Data ----------
cleaned_Testrating = cleaned_Testrating[cleaned_Testrating['rating'] >= 4]
test_recipes = cleaned_recipes[cleaned_recipes['recipe_id'].isin(cleaned_Testrating['recipe_id'])]

# ---------- Step 6: Create TF-IDF Vectors for Testing Recipes ----------
test_recipe_profiles = vectorizer.transform(test_recipes['profile'])

# ---------- Step 7: Generate Recommendations for Each User in the Testing Data ----------
recommendations = {}
for user_id in cleaned_Testrating['user_id'].unique():
    if user_id in user_profiles:
        user_profile = user_profiles[user_id]
        # Compute cosine similarity between the user's profile and test recipes' TF-IDF vectors
        similarity_scores = cosine_similarity(user_profile.reshape(1, -1), test_recipe_profiles)
        test_recipes_copy = test_recipes.copy()  # Avoid modifying the original DataFrame
        test_recipes_copy.loc[:, 'similarity'] = similarity_scores.flatten()
        recommended_recipes = test_recipes_copy.sort_values(by='similarity', ascending=False)
        top_recommendations = recommended_recipes.head(10)
        recommendations[user_id] = top_recommendations[['recipe_id', 'title', 'similarity']]

for user_id in list(recommendations.keys())[:10]:
    print("Recommendations for user", user_id)
    print(recommendations[user_id])


Recommendations for user 5215572
       recipe_id                                     title  similarity
25240      25264                          Beef Bourguignon    0.623914
9128        9138                       Peanut Butter Balls    0.494327
12670      12681                    Pumpkin Custard Or Pie    0.447196
2107        2111  Old Time Molasses Dried Apple Stack Cake    0.427496
6037        6046                Bobbie'S Apple Pie Filling    0.425534
29745      29773                      Peanut Butter Treats    0.421703
3485        3490                           One Pumpkin Pie    0.418715
5759        5768                               Gingersnaps    0.418016
24334      24358                     Grandma'S Gingerbread    0.408571
9186        9196                         Molasses Crinkles    0.405938
Recommendations for user 3622615
       recipe_id                                          title  similarity
17738      17758                           Golden Coconut Balls    1.000000
3

In [15]:
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Step 8: Calculate Root Mean Squared Error (RMSE) ---

# Initialize lists to store predicted and actual ratings
predicted_ratings = []
actual_ratings = []

# Reset indices in test_recipes to ensure alignment with test_recipe_profiles
test_recipes = test_recipes.reset_index(drop=True)

# Process users in batches to reduce memory usage
batch_size = 1000  # Adjust this based on your system's memory
user_ids = cleaned_Testrating['user_id'].unique()
num_batches = (len(user_ids) // batch_size) + 1

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min((batch_idx + 1) * batch_size, len(user_ids))
    batch_user_ids = user_ids[start_idx:end_idx]
    
    for user_id in batch_user_ids:
        if user_id in user_profiles:  # Only process users with a profile from training data
            user_profile = user_profiles[user_id].reshape(1, -1)
            
            # Get the test recipes rated by this user
            user_test_ratings = cleaned_Testrating[cleaned_Testrating['user_id'] == user_id]
            user_test_recipe_ids = user_test_ratings['recipe_id'].values
            
            # Filter test_recipe_profiles to only include recipes rated by this user
            user_test_indices = test_recipes[test_recipes['recipe_id'].isin(user_test_recipe_ids)].index
            
            # Check if indices are within the valid range
            if max(user_test_indices) >= test_recipe_profiles.shape[0]:
                print(f"Error: Indices out of range for user {user_id}. Skipping this user.")
                continue
            
            user_test_profiles = test_recipe_profiles[user_test_indices]
            
            # Compute cosine similarity between the user's profile and the filtered test recipes
            similarity_scores = cosine_similarity(user_profile, user_test_profiles).flatten()
            
            # Append predicted and actual ratings
            predicted_ratings.extend(similarity_scores)
            actual_ratings.extend(user_test_ratings['rating'].values)

# Calculate Root Mean Squared Error (RMSE)
if len(predicted_ratings) > 0 and len(actual_ratings) > 0:
    mse = mean_squared_error(actual_ratings, predicted_ratings)
    rmse = np.sqrt(mse)  # Calculate RMSE
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
else:
    print("No overlapping ratings found for RMSE calculation.")

Root Mean Squared Error (RMSE): 4.6264


In [17]:
ground_truth = {}
for user_id in cleaned_Testrating['user_id'].unique():
    ground_truth[user_id] = cleaned_Testrating[cleaned_Testrating['user_id'] == user_id]['recipe_id'].values

In [18]:
def get_recommendations(user_id):
    if user_id in recommendations:
        print(f"Recommendations for user {user_id}:")
        return recommendations[user_id]
    else:
        print(f"No recommendations available for user {user_id}.")
        return None 

user_id = 5215572 
recommended_recipes = get_recommendations(user_id)

if recommended_recipes is not None:
    print(recommended_recipes)


Recommendations for user 5215572:
       recipe_id                                     title  similarity
25240      25264                          Beef Bourguignon    0.623914
9128        9138                       Peanut Butter Balls    0.494327
12670      12681                    Pumpkin Custard Or Pie    0.447196
2107        2111  Old Time Molasses Dried Apple Stack Cake    0.427496
6037        6046                Bobbie'S Apple Pie Filling    0.425534
29745      29773                      Peanut Butter Treats    0.421703
3485        3490                           One Pumpkin Pie    0.418715
5759        5768                               Gingersnaps    0.418016
24334      24358                     Grandma'S Gingerbread    0.408571
9186        9196                         Molasses Crinkles    0.405938


In [19]:
global_TP = 0
global_FP = 0
global_FN = 0

for user_id in recommendations:
    recommended_recipe_ids = recommendations[user_id]['recipe_id'].values
    # Get the ground truth recipe IDs for the user (ensure ground_truth is a dictionary mapping user_id to recipe IDs)
    true_recipe_ids = ground_truth[user_id]
    
    overlap = set(recommended_recipe_ids) & set(true_recipe_ids)
    TP = len(overlap)
    FP = len(set(recommended_recipe_ids) - overlap)
    FN = len(set(true_recipe_ids) - overlap)
    
    global_TP += TP
    global_FP += FP
    global_FN += FN

precision_all = global_TP / (global_TP + global_FP) if (global_TP + global_FP) > 0 else 0
recall_all    = global_TP / (global_TP + global_FN) if (global_TP + global_FN) > 0 else 0
accuracy_all  = global_TP / (global_TP + global_FP + global_FN) if (global_TP + global_FP + global_FN) > 0 else 0

print(f"Global Precision: {precision_all:.4f}")
print(f"Global Recall: {recall_all:.4f}")
print(f"Global Accuracy: {accuracy_all:.4f}")


Global Precision: 0.0002
Global Recall: 0.0005
Global Accuracy: 0.0001
